# Statistical Arbitrage: Pairs Trading with Cointegration

**Market-Neutral Statistical Arbitrage**

This notebook demonstrates **cointegration-based pairs trading**, a market-neutral statistical arbitrage strategy that exploits mean-reversion in price spreads between cointegrated assets.

## What You'll Learn

1. **Pair Selection**: Correlation analysis and cointegration testing
2. **Cointegration Tests**: Engle-Granger and Johansen methodologies
3. **Spread Construction**: Hedge ratios and z-score calculation
4. **Entry/Exit Rules**: Threshold-based trading signals
5. **Position Sizing**: Dollar-neutral pairs with volatility scaling
6. **Portfolio Diversification**: Multiple pairs for risk reduction
7. **Risk Management**: Half-life monitoring and correlation breakdown
8. **Performance Analysis**: Comparison vs single-stock strategies

## Key Concepts

**Cointegration**: Two non-stationary price series that have a stationary linear combination (spread)

**Pairs Trading**: Long undervalued asset, short overvalued asset, profit when spread mean-reverts

**Market-Neutral**: Dollar-neutral positions eliminate market exposure (beta ≈ 0)

**Paper References**:
- Engle & Granger (1987): "Co-integration and Error Correction"
- Johansen (1988): "Statistical Analysis of Cointegration Vectors"
- Gatev, Goetzmann & Rouwenhorst (2006): "Pairs Trading: Performance of a Relative-Value Arbitrage Rule"
- Do & Faff (2010): "Does Simple Pairs Trading Still Work?"
- 2025 Research: Copula-based cointegration for crypto pairs (arXiv:2109.10662)

---

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Standard imports
import numpy as np
import pandas as pd
import polars as pl
from datetime import date, timedelta
from typing import List, Tuple, Dict, Optional

# Statistical tests
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import scipy.stats as stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Setup complete!")

---

## Part 1: Generate Mock Stock Data

We'll create synthetic data for 10 S&P 500 stocks with:
- **Sector structure**: Stocks within sectors are correlated
- **Cointegrated pairs**: Some pairs have long-run equilibrium relationships
- **Mean-reverting spreads**: Spreads oscillate around stable mean

In [ ]:
# Define stocks and sectors
stocks = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'GOOGL': 'Technology',
    'NVDA': 'Technology',
    'JPM': 'Financials',
    'BAC': 'Financials',
    'GS': 'Financials',
    'XOM': 'Energy',
    'CVX': 'Energy',
    'COP': 'Energy'
}

tickers = list(stocks.keys())
sectors = list(set(stocks.values()))

# Generate 504 days of data (2 years, ~252 trading days/year)
n_days = 504
dates = [date(2022, 1, 1) + timedelta(days=i) for i in range(n_days)]

print(f"✓ Generating {n_days} days of price data for {len(tickers)} stocks")
print(f"\nStocks by sector:")
for sector in sectors:
    sector_stocks = [t for t, s in stocks.items() if s == sector]
    print(f"  {sector}: {', '.join(sector_stocks)}")

In [ ]:
def generate_cointegrated_prices(n_days: int, stocks_dict: Dict[str, str],
                                 sector_correlation: float = 0.7,
                                 cointegration_strength: float = 0.9) -> Dict[str, np.ndarray]:
    """
    Generate price series with cointegration within sectors.
    
    Strategy:
    1. Generate sector-level common factor (random walk)
    2. Each stock = sector_factor + idiosyncratic_factor
    3. Within sector, stocks share cointegration relationship
    4. Between sectors, stocks are correlated but not cointegrated
    
    Args:
        n_days: Number of trading days
        stocks_dict: {ticker: sector} mapping
        sector_correlation: Correlation strength within sectors
        cointegration_strength: How strongly pairs cointegrate (0-1)
    
    Returns:
        Dictionary {ticker: price_series}
    """
    # Group stocks by sector
    sector_stocks = {}
    for ticker, sector in stocks_dict.items():
        if sector not in sector_stocks:
            sector_stocks[sector] = []
        sector_stocks[sector].append(ticker)
    
    prices = {}
    
    # Generate prices for each sector
    for sector, sector_tickers in sector_stocks.items():
        n_stocks = len(sector_tickers)
        
        # Generate common sector factor (non-stationary random walk)
        sector_shocks = np.random.normal(0, 0.015, n_days)
        sector_factor = 100 + np.cumsum(sector_shocks)
        
        # Generate mean-reverting component (stationary)
        # AR(1) process: x_t = φ * x_{t-1} + ε_t, where φ < 1
        phi = 0.95  # Mean reversion speed
        mean_reverting = np.zeros(n_days)
        for t in range(1, n_days):
            shock = np.random.normal(0, 0.5)
            mean_reverting[t] = phi * mean_reverting[t-1] + shock
        
        # Each stock = sector_factor + stock_specific_drift + mean_reverting_component
        for i, ticker in enumerate(sector_tickers):
            # Stock-specific drift (small)
            drift = np.random.normal(0, 0.001, n_days).cumsum()
            
            # Idiosyncratic shocks
            idiosyncratic = np.random.normal(0, 0.005, n_days).cumsum()
            
            # Combine: cointegrated = shared sector + small idiosyncratic
            # cointegration_strength controls how much mean-reverting vs random walk
            stock_price = (
                cointegration_strength * sector_factor +
                (1 - cointegration_strength) * idiosyncratic +
                0.3 * mean_reverting[i::n_stocks][:n_days] +  # Staggered mean reversion
                drift
            )
            
            # Ensure positive prices
            stock_price = np.maximum(stock_price, 10)
            
            prices[ticker] = stock_price
    
    return prices

# Generate cointegrated prices
prices_dict = generate_cointegrated_prices(
    n_days=n_days,
    stocks_dict=stocks,
    sector_correlation=0.7,
    cointegration_strength=0.85
)

# Create DataFrame
prices_data = []
for ticker in tickers:
    for i, d in enumerate(dates):
        prices_data.append({
            'date': d,
            'ticker': ticker,
            'price': prices_dict[ticker][i],
            'sector': stocks[ticker]
        })

prices_df = pl.DataFrame(prices_data)

# Calculate returns
returns_df = prices_df.sort(['ticker', 'date']).with_columns(
    pl.col('price').pct_change().over('ticker').alias('return')
).drop_nulls()

print("\n✓ Generated cointegrated price series")
print(f"Price range: ${prices_df['price'].min():.2f} - ${prices_df['price'].max():.2f}")
print(f"Mean daily return: {returns_df['return'].mean():.4%}")
print(f"Daily volatility: {returns_df['return'].std():.4%}")

In [ ]:
# Visualize price series by sector
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i, sector in enumerate(sectors):
    sector_tickers = [t for t, s in stocks.items() if s == sector]
    
    for ticker in sector_tickers:
        ticker_data = prices_df.filter(pl.col('ticker') == ticker).sort('date')
        axes[i].plot(ticker_data['date'].to_list(), 
                    ticker_data['price'].to_list(),
                    label=ticker, linewidth=2, alpha=0.7)
    
    axes[i].set_title(f'{sector} Sector - Price Evolution', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Price ($)', fontsize=10)
    axes[i].legend(loc='upper left')
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Date', fontsize=10)
plt.tight_layout()
plt.show()

print("\n💡 Observation: Stocks within the same sector move together (cointegration)")

---

## Part 2: Pair Selection via Correlation Analysis

**Step 1: Calculate pairwise correlations**

Correlation is a necessary (but not sufficient) condition for cointegration:
- High correlation (ρ > 0.7) suggests potential cointegration
- Must verify with formal cointegration tests
- Spurious correlation is common → always test for cointegration!

In [ ]:
# Create wide-format price matrix for correlation
prices_wide = prices_df.pivot(
    index='date',
    columns='ticker',
    values='price'
).to_pandas()

# Calculate correlation matrix
corr_matrix = prices_wide.corr()

# Visualize correlation heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Mask upper triangle
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5,
            vmin=0, vmax=1, square=True, mask=mask, cbar_kws={'label': 'Correlation'})
plt.title('Price Correlation Matrix (Lower Triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Correlation Analysis:")
print("High correlation within sectors (diagonal blocks) suggests potential cointegration")

In [ ]:
# Identify highly correlated pairs
def find_correlated_pairs(corr_matrix: pd.DataFrame, 
                          min_correlation: float = 0.7) -> List[Tuple[str, str, float]]:
    """
    Find pairs with correlation above threshold.
    
    Args:
        corr_matrix: Correlation matrix
        min_correlation: Minimum correlation to consider
    
    Returns:
        List of (ticker_A, ticker_B, correlation) tuples
    """
    pairs = []
    tickers = corr_matrix.columns.tolist()
    
    for i, ticker_A in enumerate(tickers):
        for ticker_B in tickers[i+1:]:
            corr = corr_matrix.loc[ticker_A, ticker_B]
            if corr >= min_correlation:
                pairs.append((ticker_A, ticker_B, corr))
    
    # Sort by correlation (descending)
    pairs.sort(key=lambda x: x[2], reverse=True)
    
    return pairs

# Find pairs with correlation > 0.70
candidate_pairs = find_correlated_pairs(corr_matrix, min_correlation=0.70)

print(f"✓ Found {len(candidate_pairs)} candidate pairs with correlation ≥ 0.70\n")
print("Top 10 Correlated Pairs:")
print("=" * 60)
print(f"{'Pair':<15} {'Sector A':<12} {'Sector B':<12} {'Correlation'}")
print("=" * 60)

for ticker_A, ticker_B, corr in candidate_pairs[:10]:
    sector_A = stocks[ticker_A]
    sector_B = stocks[ticker_B]
    same_sector = "✓" if sector_A == sector_B else " "
    print(f"{ticker_A}-{ticker_B:<10} {sector_A:<12} {sector_B:<12} {corr:.4f} {same_sector}")

print("\n✓ = Same sector (more likely to be cointegrated)")

---

## Part 3: Cointegration Testing

### 3.1 Engle-Granger Two-Step Method

**Methodology** (Engle & Granger, 1987):

1. **Step 1**: Regress price_A on price_B to estimate hedge ratio
   - `price_A = α + β * price_B + ε`
   - Hedge ratio: `β`

2. **Step 2**: Test if residuals (spread) are stationary using ADF test
   - Spread: `spread = price_A - β * price_B`
   - ADF test: H₀ = spread has unit root (non-stationary)
   - If p-value < 0.05 → reject H₀ → spread is stationary → cointegrated!

**Critical Values** (Engle-Granger):
- 1% level: -3.90
- 5% level: -3.34
- 10% level: -3.04

In [ ]:
def engle_granger_test(price_A: np.ndarray, price_B: np.ndarray,
                       significance_level: float = 0.05) -> Dict:
    """
    Engle-Granger two-step cointegration test.
    
    Args:
        price_A: Price series for asset A
        price_B: Price series for asset B
        significance_level: Significance level for test (default: 0.05)
    
    Returns:
        Dictionary with test results:
        - cointegrated: bool
        - t_statistic: ADF test statistic
        - p_value: p-value
        - hedge_ratio: β coefficient
        - spread: residual series
    """
    # Use statsmodels coint function (performs Engle-Granger test)
    t_stat, p_value, crit_values = coint(price_A, price_B)
    
    # Estimate hedge ratio via OLS
    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    model.fit(price_B.reshape(-1, 1), price_A)
    hedge_ratio = model.coef_[0]
    
    # Calculate spread
    spread = price_A - hedge_ratio * price_B
    
    # Cointegrated if p-value < significance level
    cointegrated = p_value < significance_level
    
    return {
        'cointegrated': cointegrated,
        't_statistic': t_stat,
        'p_value': p_value,
        'hedge_ratio': hedge_ratio,
        'spread': spread,
        'critical_values': crit_values
    }

# Test all candidate pairs
engle_granger_results = []

for ticker_A, ticker_B, corr in candidate_pairs:
    # Extract price series
    price_A = prices_wide[ticker_A].values
    price_B = prices_wide[ticker_B].values
    
    # Run Engle-Granger test
    result = engle_granger_test(price_A, price_B)
    
    engle_granger_results.append({
        'ticker_A': ticker_A,
        'ticker_B': ticker_B,
        'correlation': corr,
        'cointegrated': result['cointegrated'],
        't_statistic': result['t_statistic'],
        'p_value': result['p_value'],
        'hedge_ratio': result['hedge_ratio'],
        'spread': result['spread']
    })

# Create results DataFrame
eg_results_df = pd.DataFrame(engle_granger_results)

# Filter cointegrated pairs
cointegrated_pairs = eg_results_df[eg_results_df['cointegrated']]

print("\n" + "=" * 80)
print("ENGLE-GRANGER COINTEGRATION TEST RESULTS")
print("=" * 80)
print(f"\nTotal pairs tested: {len(eg_results_df)}")
print(f"Cointegrated pairs: {len(cointegrated_pairs)} ({len(cointegrated_pairs)/len(eg_results_df)*100:.1f}%)\n")

print("Cointegrated Pairs (p-value < 0.05):")
print("=" * 80)
print(f"{'Pair':<15} {'Correlation':<12} {'Hedge Ratio':<13} {'T-Stat':<10} {'P-Value'}")
print("=" * 80)

for _, row in cointegrated_pairs.iterrows():
    pair = f"{row['ticker_A']}-{row['ticker_B']}"
    print(f"{pair:<15} {row['correlation']:<12.4f} {row['hedge_ratio']:<13.4f} "
          f"{row['t_statistic']:<10.4f} {row['p_value']:.4f}")

if len(cointegrated_pairs) == 0:
    print("⚠️  No cointegrated pairs found at 5% significance level")
    print("This is expected with purely synthetic data. Real markets show ~10-20% cointegration rate.")

### 3.2 Johansen Test (Multivariate)

**Methodology** (Johansen, 1988):

The Johansen test is more powerful than Engle-Granger:
- Tests for multiple cointegrating relationships simultaneously
- Not sensitive to choice of dependent variable
- Can test N > 2 assets at once

**Test Statistics**:
- **Trace statistic**: Tests H₀: "at most r cointegrating vectors"
- **Max eigenvalue**: Tests H₀: "exactly r cointegrating vectors"

We focus on pairwise tests (r = 1) for pairs trading.

In [ ]:
def johansen_test(price_A: np.ndarray, price_B: np.ndarray,
                 significance_level: str = '5%') -> Dict:
    """
    Johansen cointegration test for two assets.
    
    Args:
        price_A: Price series for asset A
        price_B: Price series for asset B
        significance_level: '1%', '5%', or '10%'
    
    Returns:
        Dictionary with test results
    """
    # Prepare data matrix [price_A, price_B]
    data = np.column_stack([price_A, price_B])
    
    # Run Johansen test (det_order=-1 means no deterministic trend)
    result = coint_johansen(data, det_order=0, k_ar_diff=1)
    
    # Extract trace statistic and critical value
    trace_stat = result.lr1[0]  # Trace statistic for r=0
    
    # Critical values: 90%, 95%, 99%
    sig_map = {'10%': 0, '5%': 1, '1%': 2}
    crit_idx = sig_map.get(significance_level, 1)
    critical_value = result.cvt[0, crit_idx]
    
    # Cointegrated if trace_stat > critical_value
    cointegrated = trace_stat > critical_value
    
    # Cointegrating vector (hedge ratio)
    coint_vector = result.evec[:, 0]  # First eigenvector
    hedge_ratio = -coint_vector[1] / coint_vector[0]  # Normalize
    
    return {
        'cointegrated': cointegrated,
        'trace_statistic': trace_stat,
        'critical_value': critical_value,
        'hedge_ratio': hedge_ratio,
        'eigenvector': coint_vector
    }

# Test all candidate pairs with Johansen
johansen_results = []

for ticker_A, ticker_B, corr in candidate_pairs:
    # Extract price series
    price_A = prices_wide[ticker_A].values
    price_B = prices_wide[ticker_B].values
    
    # Run Johansen test
    result = johansen_test(price_A, price_B, significance_level='5%')
    
    johansen_results.append({
        'ticker_A': ticker_A,
        'ticker_B': ticker_B,
        'correlation': corr,
        'cointegrated': result['cointegrated'],
        'trace_statistic': result['trace_statistic'],
        'critical_value': result['critical_value'],
        'hedge_ratio': result['hedge_ratio']
    })

# Create results DataFrame
johansen_results_df = pd.DataFrame(johansen_results)

# Filter cointegrated pairs
johansen_cointegrated = johansen_results_df[johansen_results_df['cointegrated']]

print("\n" + "=" * 80)
print("JOHANSEN COINTEGRATION TEST RESULTS")
print("=" * 80)
print(f"\nTotal pairs tested: {len(johansen_results_df)}")
print(f"Cointegrated pairs: {len(johansen_cointegrated)} ({len(johansen_cointegrated)/len(johansen_results_df)*100:.1f}%)\n")

print("Cointegrated Pairs (Trace > Critical Value at 5%):")
print("=" * 85)
print(f"{'Pair':<15} {'Correlation':<12} {'Hedge Ratio':<13} {'Trace':<10} {'Critical'}")
print("=" * 85)

for _, row in johansen_cointegrated.iterrows():
    pair = f"{row['ticker_A']}-{row['ticker_B']}"
    print(f"{pair:<15} {row['correlation']:<12.4f} {row['hedge_ratio']:<13.4f} "
          f"{row['trace_statistic']:<10.4f} {row['critical_value']:.4f}")

if len(johansen_cointegrated) == 0:
    print("⚠️  No cointegrated pairs found at 5% significance level")
    print("This can occur with synthetic data. We'll proceed with highest-correlation pairs for demonstration.")

In [ ]:
# Compare Engle-Granger vs Johansen
print("\n" + "=" * 80)
print("COMPARISON: ENGLE-GRANGER vs JOHANSEN")
print("=" * 80)

# Merge results
comparison = eg_results_df.merge(
    johansen_results_df[['ticker_A', 'ticker_B', 'cointegrated', 'hedge_ratio']],
    on=['ticker_A', 'ticker_B'],
    suffixes=('_EG', '_Johansen')
)

# Count agreements
both_yes = ((comparison['cointegrated_EG']) & (comparison['cointegrated_Johansen'])).sum()
both_no = ((~comparison['cointegrated_EG']) & (~comparison['cointegrated_Johansen'])).sum()
disagree = len(comparison) - both_yes - both_no

print(f"\nAgreement Rate: {(both_yes + both_no) / len(comparison) * 100:.1f}%")
print(f"  Both cointegrated: {both_yes}")
print(f"  Both not cointegrated: {both_no}")
print(f"  Disagree: {disagree}")

print("\n💡 Key Insights:")
print("  • Johansen test is generally more robust than Engle-Granger")
print("  • Engle-Granger is simpler and more widely used in practice")
print("  • Use both tests for confirmation of strong cointegration")

---

## Part 4: Spread Construction and Mean-Reversion Analysis

For cointegrated pairs, we construct the spread:

**Spread = Price_A - β × Price_B**

where β is the hedge ratio (cointegration coefficient)

The spread should be:
- **Stationary** (verified by cointegration tests)
- **Mean-reverting** (oscillates around long-run mean)
- **Tradeable** (reasonable half-life of mean reversion)

In [ ]:
# Select pairs for trading (use top pairs by correlation if no cointegration found)
if len(cointegrated_pairs) > 0:
    trading_pairs_df = cointegrated_pairs.head(5)
    print("✓ Using cointegrated pairs for trading")
else:
    # Fallback: use top 5 correlated pairs
    trading_pairs_df = eg_results_df.nlargest(5, 'correlation')
    print("⚠️  Using highest-correlation pairs (cointegration not confirmed)")
    print("In production, only trade confirmed cointegrated pairs!")

print(f"\n✓ Selected {len(trading_pairs_df)} pairs for trading\n")

# Calculate spreads for each pair
spreads = {}
for _, row in trading_pairs_df.iterrows():
    ticker_A = row['ticker_A']
    ticker_B = row['ticker_B']
    hedge_ratio = row['hedge_ratio']
    
    price_A = prices_wide[ticker_A].values
    price_B = prices_wide[ticker_B].values
    
    spread = price_A - hedge_ratio * price_B
    spreads[f"{ticker_A}-{ticker_B}"] = {
        'spread': spread,
        'hedge_ratio': hedge_ratio,
        'ticker_A': ticker_A,
        'ticker_B': ticker_B
    }

print("✓ Calculated spreads for all trading pairs")

In [ ]:
# Visualize spreads
n_pairs = len(spreads)
fig, axes = plt.subplots(n_pairs, 1, figsize=(14, 4 * n_pairs))

if n_pairs == 1:
    axes = [axes]

for idx, (pair_name, pair_data) in enumerate(spreads.items()):
    spread = pair_data['spread']
    hedge_ratio = pair_data['hedge_ratio']
    
    # Plot spread
    axes[idx].plot(dates, spread, linewidth=2, color='steelblue', label='Spread')
    
    # Plot mean and ±2σ bands
    mean = np.mean(spread)
    std = np.std(spread)
    
    axes[idx].axhline(mean, color='black', linestyle='--', linewidth=1.5, label='Mean')
    axes[idx].axhline(mean + 2*std, color='red', linestyle='--', linewidth=1, label='+2σ')
    axes[idx].axhline(mean - 2*std, color='green', linestyle='--', linewidth=1, label='-2σ')
    
    axes[idx].fill_between(dates, mean - 2*std, mean + 2*std, alpha=0.2, color='gray')
    
    axes[idx].set_title(f'{pair_name} Spread (β={hedge_ratio:.4f})', 
                       fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Spread ($)', fontsize=10)
    axes[idx].legend(loc='upper left')
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Date', fontsize=10)
plt.tight_layout()
plt.show()

print("\n💡 Trading Logic:")
print("  • When spread > +2σ: Spread is HIGH → Short spread (sell A, buy B)")
print("  • When spread < -2σ: Spread is LOW → Long spread (buy A, sell B)")
print("  • Exit when spread crosses mean (z-score ≈ 0)")

### 4.1 Half-Life of Mean Reversion

**Half-life** measures how long it takes for the spread to revert halfway to its mean.

**Calculation via AR(1) model**:
- Fit: `spread_t = φ × spread_{t-1} + ε`
- Half-life: `τ = -ln(2) / ln(φ)`

**Rule of thumb**:
- τ < 10 days: Fast mean reversion (good for trading)
- 10 ≤ τ ≤ 30 days: Moderate mean reversion (acceptable)
- τ > 30 days: Slow mean reversion (risky, may not converge)

In [ ]:
def calculate_half_life(spread: np.ndarray) -> float:
    """
    Calculate half-life of mean reversion using AR(1) model.
    
    Args:
        spread: Time series of spread values
    
    Returns:
        Half-life in number of periods (days)
    """
    # Fit AR(1): spread_t = φ * spread_{t-1} + ε
    spread_lag = spread[:-1]
    spread_current = spread[1:]
    
    # OLS regression
    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    model.fit(spread_lag.reshape(-1, 1), spread_current)
    
    phi = model.coef_[0]
    
    # Half-life formula
    if phi >= 1 or phi <= 0:
        return np.inf  # No mean reversion
    
    half_life = -np.log(2) / np.log(phi)
    
    return half_life

# Calculate half-life for each pair
print("\n" + "=" * 70)
print("HALF-LIFE ANALYSIS")
print("=" * 70)
print(f"\n{'Pair':<15} {'Half-Life (days)':<18} {'Assessment'}")
print("=" * 70)

for pair_name, pair_data in spreads.items():
    spread = pair_data['spread']
    half_life = calculate_half_life(spread)
    
    # Store in dict
    pair_data['half_life'] = half_life
    
    # Assessment
    if half_life < 10:
        assessment = "✓ Fast (Good)"
    elif half_life < 30:
        assessment = "○ Moderate (OK)"
    else:
        assessment = "✗ Slow (Risky)"
    
    print(f"{pair_name:<15} {half_life:<18.2f} {assessment}")

print("\n💡 Interpretation:")
print("  • Shorter half-life = faster mean reversion = better for pairs trading")
print("  • Typical tradeable pairs have half-life < 30 days")

---

## Part 5: Z-Score Based Entry/Exit Signals

**Z-Score Calculation**:
```
z_score = (spread - rolling_mean) / rolling_std
```

**Trading Rules**:
- **Entry Long**: z < -2.0 (spread 2σ below mean)
- **Entry Short**: z > +2.0 (spread 2σ above mean)
- **Exit**: z crosses 0 (spread returns to mean)
- **Stop-Loss**: z > 3.0 (correlation breakdown)

**Dollar-Neutral Positions**:
- If long spread: Long $X of A, Short $X×β of B
- If short spread: Short $X of A, Long $X×β of B

In [ ]:
def calculate_z_score(spread: np.ndarray, window: int = 20) -> np.ndarray:
    """
    Calculate rolling z-score for spread.
    
    Args:
        spread: Time series of spread values
        window: Rolling window size (default: 20 days)
    
    Returns:
        Array of z-scores
    """
    # Convert to pandas for rolling operations
    spread_series = pd.Series(spread)
    
    rolling_mean = spread_series.rolling(window=window, min_periods=window).mean()
    rolling_std = spread_series.rolling(window=window, min_periods=window).std()
    
    z_score = (spread_series - rolling_mean) / rolling_std
    
    return z_score.values

def generate_signals(z_score: np.ndarray, 
                    entry_threshold: float = 2.0,
                    exit_threshold: float = 0.0,
                    stop_loss: float = 3.0) -> np.ndarray:
    """
    Generate trading signals from z-scores.
    
    Args:
        z_score: Array of z-scores
        entry_threshold: Z-score threshold for entry (default: 2.0)
        exit_threshold: Z-score threshold for exit (default: 0.0)
        stop_loss: Z-score threshold for stop-loss (default: 3.0)
    
    Returns:
        Array of positions: +1 (long spread), -1 (short spread), 0 (flat)
    """
    positions = np.zeros(len(z_score))
    current_position = 0
    
    for i in range(len(z_score)):
        if np.isnan(z_score[i]):
            positions[i] = current_position
            continue
        
        # Entry signals
        if current_position == 0:
            if z_score[i] > entry_threshold:
                current_position = -1  # Short spread (sell A, buy B)
            elif z_score[i] < -entry_threshold:
                current_position = 1   # Long spread (buy A, sell B)
        
        # Exit signals
        elif current_position == 1:  # Currently long
            if z_score[i] > exit_threshold or z_score[i] > stop_loss:
                current_position = 0
        
        elif current_position == -1:  # Currently short
            if z_score[i] < exit_threshold or z_score[i] < -stop_loss:
                current_position = 0
        
        positions[i] = current_position
    
    return positions

# Generate signals for all pairs
signals_data = {}

for pair_name, pair_data in spreads.items():
    spread = pair_data['spread']
    
    # Calculate z-score
    z_score = calculate_z_score(spread, window=20)
    
    # Generate signals
    positions = generate_signals(z_score, entry_threshold=2.0, exit_threshold=0.0, stop_loss=3.0)
    
    # Store results
    signals_data[pair_name] = {
        'z_score': z_score,
        'positions': positions,
        'spread': spread,
        'hedge_ratio': pair_data['hedge_ratio'],
        'ticker_A': pair_data['ticker_A'],
        'ticker_B': pair_data['ticker_B']
    }

print("✓ Generated z-scores and trading signals for all pairs")

In [ ]:
# Visualize signals for first pair
pair_name = list(signals_data.keys())[0]
pair_signals = signals_data[pair_name]

fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 1, height_ratios=[2, 1, 1], hspace=0.3)

# Panel 1: Spread with positions
ax1 = fig.add_subplot(gs[0])
spread = pair_signals['spread']
positions = pair_signals['positions']

ax1.plot(dates, spread, linewidth=2, color='steelblue', label='Spread')
ax1.axhline(np.mean(spread), color='black', linestyle='--', linewidth=1.5, label='Mean')

# Highlight positions
long_mask = positions == 1
short_mask = positions == -1

ax1.fill_between(dates, spread.min(), spread.max(), where=long_mask, 
                 alpha=0.3, color='green', label='Long Spread')
ax1.fill_between(dates, spread.min(), spread.max(), where=short_mask,
                 alpha=0.3, color='red', label='Short Spread')

ax1.set_title(f'{pair_name} - Trading Signals', fontsize=14, fontweight='bold')
ax1.set_ylabel('Spread ($)', fontsize=10)
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Panel 2: Z-score
ax2 = fig.add_subplot(gs[1])
z_score = pair_signals['z_score']

ax2.plot(dates, z_score, linewidth=2, color='purple', label='Z-Score')
ax2.axhline(0, color='black', linestyle='-', linewidth=1)
ax2.axhline(2, color='red', linestyle='--', linewidth=1, label='Entry Threshold')
ax2.axhline(-2, color='green', linestyle='--', linewidth=1)
ax2.axhline(3, color='darkred', linestyle=':', linewidth=1, label='Stop-Loss')
ax2.axhline(-3, color='darkgreen', linestyle=':', linewidth=1)

ax2.fill_between(dates, -2, 2, alpha=0.2, color='gray', label='No Trade Zone')

ax2.set_ylabel('Z-Score', fontsize=10)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Panel 3: Position
ax3 = fig.add_subplot(gs[2])
ax3.fill_between(dates, 0, positions, where=(positions > 0), 
                 step='mid', alpha=0.7, color='green', label='Long')
ax3.fill_between(dates, 0, positions, where=(positions < 0),
                 step='mid', alpha=0.7, color='red', label='Short')
ax3.axhline(0, color='black', linestyle='-', linewidth=1)

ax3.set_xlabel('Date', fontsize=10)
ax3.set_ylabel('Position', fontsize=10)
ax3.set_yticks([-1, 0, 1])
ax3.set_yticklabels(['Short', 'Flat', 'Long'])
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Count trades
position_changes = np.diff(positions)
n_trades = np.sum(position_changes != 0)
n_days_in_trade = np.sum(positions != 0)

print(f"\n📊 Trading Statistics for {pair_name}:")
print(f"  Number of trades: {n_trades}")
print(f"  Days in trade: {n_days_in_trade} ({n_days_in_trade/len(positions)*100:.1f}%)")
print(f"  Average trade duration: {n_days_in_trade/(n_trades/2 + 0.001):.1f} days")

---

## Part 6: Position Sizing and Portfolio Construction

**Position Sizing Principles**:

1. **Dollar-Neutral**: Long $X of A, Short $X×β of B
2. **Volatility Scaling**: Size inversely proportional to spread volatility
3. **Equal Risk Contribution**: Allocate capital such that each pair contributes equally to portfolio risk

**Formula**:
```
position_size = capital / (spread_volatility × sqrt(N_pairs))
```

In [ ]:
def calculate_pair_returns(price_A: np.ndarray, price_B: np.ndarray,
                          hedge_ratio: float, positions: np.ndarray) -> np.ndarray:
    """
    Calculate pair trading returns.
    
    Args:
        price_A: Price series for asset A
        price_B: Price series for asset B
        hedge_ratio: Hedge ratio (β)
        positions: Position array (+1, -1, or 0)
    
    Returns:
        Array of daily returns
    """
    # Calculate daily returns
    ret_A = np.diff(price_A) / price_A[:-1]
    ret_B = np.diff(price_B) / price_B[:-1]
    
    # Pair return = position × (ret_A - hedge_ratio × ret_B)
    # Note: This is dollar-neutral spread return
    pair_returns = positions[:-1] * (ret_A - hedge_ratio * ret_B)
    
    return pair_returns

# Calculate returns for each pair
pair_returns_dict = {}

for pair_name, pair_signals in signals_data.items():
    ticker_A = pair_signals['ticker_A']
    ticker_B = pair_signals['ticker_B']
    hedge_ratio = pair_signals['hedge_ratio']
    positions = pair_signals['positions']
    
    price_A = prices_wide[ticker_A].values
    price_B = prices_wide[ticker_B].values
    
    pair_returns = calculate_pair_returns(price_A, price_B, hedge_ratio, positions)
    
    pair_returns_dict[pair_name] = pair_returns

# Create returns DataFrame
returns_matrix = pd.DataFrame(pair_returns_dict, index=dates[1:])  # Drop first date

print("✓ Calculated returns for all pairs")
print(f"\nReturns matrix shape: {returns_matrix.shape}")
print(f"Date range: {returns_matrix.index[0]} to {returns_matrix.index[-1]}")

In [ ]:
# Equal-weight portfolio of pairs
portfolio_returns_equal = returns_matrix.mean(axis=1)

# Volatility-weighted portfolio
pair_volatilities = returns_matrix.std()
inverse_vol_weights = (1 / pair_volatilities) / (1 / pair_volatilities).sum()
portfolio_returns_vol_weighted = (returns_matrix * inverse_vol_weights).sum(axis=1)

print("\n📊 Portfolio Construction:")
print("=" * 60)
print(f"\n{'Pair':<15} {'Volatility':<12} {'Equal Weight':<13} {'Vol Weight'}")
print("=" * 60)

equal_weight = 1 / len(returns_matrix.columns)
for pair_name in returns_matrix.columns:
    vol = pair_volatilities[pair_name]
    vol_weight = inverse_vol_weights[pair_name]
    print(f"{pair_name:<15} {vol:<12.4%} {equal_weight:<13.2%} {vol_weight:.2%}")

print("\n💡 Volatility weighting reduces exposure to high-volatility pairs")

---

## Part 7: Correlation Breakdown Detection

**Risk**: Cointegration can break down over time!

**Monitoring**:
1. **Rolling correlation**: Track 60-day correlation
2. **Half-life monitoring**: Recalculate half-life every 30 days
3. **Cointegration re-test**: Run Engle-Granger test periodically

**Exit rules if correlation breaks down**:
- Correlation drops below 0.5
- Half-life exceeds 60 days
- Spread exceeds 3σ (stop-loss)

In [ ]:
# Calculate rolling correlation for each pair
def calculate_rolling_correlation(price_A: np.ndarray, price_B: np.ndarray,
                                 window: int = 60) -> np.ndarray:
    """
    Calculate rolling correlation between two price series.
    
    Args:
        price_A: Price series for asset A
        price_B: Price series for asset B
        window: Rolling window size (default: 60 days)
    
    Returns:
        Array of rolling correlations
    """
    series_A = pd.Series(price_A)
    series_B = pd.Series(price_B)
    
    rolling_corr = series_A.rolling(window=window).corr(series_B)
    
    return rolling_corr.values

# Calculate rolling correlation for first pair
pair_name = list(signals_data.keys())[0]
pair_signals = signals_data[pair_name]

ticker_A = pair_signals['ticker_A']
ticker_B = pair_signals['ticker_B']

price_A = prices_wide[ticker_A].values
price_B = prices_wide[ticker_B].values

rolling_corr = calculate_rolling_correlation(price_A, price_B, window=60)

# Visualize correlation stability
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Panel 1: Rolling correlation
axes[0].plot(dates, rolling_corr, linewidth=2, color='steelblue', label='60-Day Rolling Correlation')
axes[0].axhline(0.7, color='green', linestyle='--', linewidth=1.5, label='Min Threshold (0.7)')
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Breakdown Warning (0.5)')
axes[0].fill_between(dates, 0.5, 0.7, alpha=0.2, color='yellow', label='Warning Zone')
axes[0].fill_between(dates, 0, 0.5, alpha=0.3, color='red', label='Breakdown Zone')

axes[0].set_title(f'{pair_name} - Correlation Stability Monitoring', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Correlation', fontsize=10)
axes[0].set_ylim([0, 1])
axes[0].legend(loc='lower left')
axes[0].grid(True, alpha=0.3)

# Panel 2: Cumulative returns with correlation breakdown zones
pair_returns = pair_returns_dict[pair_name]
cum_returns = (1 + pd.Series(pair_returns, index=dates[1:])).cumprod()

axes[1].plot(cum_returns.index, cum_returns.values, linewidth=2, color='green', label='Cumulative Returns')
axes[1].axhline(1, color='black', linestyle='--', linewidth=1)

# Highlight periods of low correlation
low_corr_mask = rolling_corr[1:] < 0.5
axes[1].fill_between(dates[1:], cum_returns.min(), cum_returns.max(),
                     where=low_corr_mask, alpha=0.3, color='red',
                     label='Correlation Breakdown')

axes[1].set_xlabel('Date', fontsize=10)
axes[1].set_ylabel('Cumulative Return', fontsize=10)
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
breakdown_days = np.sum(rolling_corr < 0.5)
warning_days = np.sum((rolling_corr >= 0.5) & (rolling_corr < 0.7))

print(f"\n📊 Correlation Stability for {pair_name}:")
print(f"  Days with correlation < 0.5 (breakdown): {breakdown_days} ({breakdown_days/len(rolling_corr)*100:.1f}%)")
print(f"  Days with 0.5 ≤ correlation < 0.7 (warning): {warning_days} ({warning_days/len(rolling_corr)*100:.1f}%)")
print(f"  Mean rolling correlation: {np.nanmean(rolling_corr):.3f}")
print(f"  Std rolling correlation: {np.nanstd(rolling_corr):.3f}")

---

## Part 8: Performance Analysis

### 8.1 Pairs Portfolio Performance

In [ ]:
# Calculate performance metrics
def calculate_performance_metrics(returns: pd.Series) -> Dict:
    """
    Calculate performance metrics for a return series.
    
    Args:
        returns: Daily returns series
    
    Returns:
        Dictionary of metrics
    """
    total_return = (1 + returns).prod() - 1
    annualized_return = (1 + total_return) ** (252 / len(returns)) - 1
    annualized_vol = returns.std() * np.sqrt(252)
    sharpe_ratio = annualized_return / annualized_vol if annualized_vol > 0 else 0
    
    # Max drawdown
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.expanding().max()
    drawdown = (cum_returns - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Calmar ratio
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # Win rate
    win_rate = (returns > 0).sum() / len(returns)
    
    return {
        'total_return': total_return,
        'annualized_return': annualized_return,
        'annualized_vol': annualized_vol,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'win_rate': win_rate
    }

# Calculate metrics for portfolios
equal_weight_metrics = calculate_performance_metrics(portfolio_returns_equal)
vol_weight_metrics = calculate_performance_metrics(portfolio_returns_vol_weighted)

# Calculate metrics for individual pairs
pair_metrics = {}
for pair_name, pair_returns in pair_returns_dict.items():
    pair_metrics[pair_name] = calculate_performance_metrics(pd.Series(pair_returns))

print("\n" + "=" * 80)
print("PAIRS TRADING PORTFOLIO PERFORMANCE")
print("=" * 80)

print("\n📊 Portfolio-Level Metrics:")
print("=" * 80)
print(f"{'Metric':<25} {'Equal-Weight':<20} {'Vol-Weight'}")
print("=" * 80)
print(f"{'Total Return':<25} {equal_weight_metrics['total_return']:<20.2%} {vol_weight_metrics['total_return']:.2%}")
print(f"{'Annualized Return':<25} {equal_weight_metrics['annualized_return']:<20.2%} {vol_weight_metrics['annualized_return']:.2%}")
print(f"{'Annualized Volatility':<25} {equal_weight_metrics['annualized_vol']:<20.2%} {vol_weight_metrics['annualized_vol']:.2%}")
print(f"{'Sharpe Ratio':<25} {equal_weight_metrics['sharpe_ratio']:<20.3f} {vol_weight_metrics['sharpe_ratio']:.3f}")
print(f"{'Max Drawdown':<25} {equal_weight_metrics['max_drawdown']:<20.2%} {vol_weight_metrics['max_drawdown']:.2%}")
print(f"{'Calmar Ratio':<25} {equal_weight_metrics['calmar_ratio']:<20.3f} {vol_weight_metrics['calmar_ratio']:.3f}")
print(f"{'Win Rate':<25} {equal_weight_metrics['win_rate']:<20.2%} {vol_weight_metrics['win_rate']:.2%}")

print("\n\n📊 Individual Pair Performance:")
print("=" * 95)
print(f"{'Pair':<15} {'Total Return':<13} {'Ann. Return':<13} {'Sharpe':<10} {'Max DD':<10} {'Win Rate'}")
print("=" * 95)

for pair_name, metrics in pair_metrics.items():
    print(f"{pair_name:<15} {metrics['total_return']:<13.2%} {metrics['annualized_return']:<13.2%} "
          f"{metrics['sharpe_ratio']:<10.3f} {metrics['max_drawdown']:<10.2%} {metrics['win_rate']:.2%}")

In [ ]:
# Visualize cumulative returns
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Cumulative returns
cum_returns_equal = (1 + portfolio_returns_equal).cumprod()
cum_returns_vol = (1 + portfolio_returns_vol_weighted).cumprod()

axes[0].plot(cum_returns_equal.index, cum_returns_equal.values, 
            linewidth=2.5, color='steelblue', label='Equal-Weight Portfolio')
axes[0].plot(cum_returns_vol.index, cum_returns_vol.values,
            linewidth=2.5, color='green', label='Vol-Weight Portfolio')
axes[0].axhline(1, color='black', linestyle='--', linewidth=1)

axes[0].set_title('Pairs Trading Portfolio - Cumulative Returns', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cumulative Return', fontsize=10)
axes[0].legend(loc='upper left', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Panel 2: Drawdown
cum_max_equal = cum_returns_equal.expanding().max()
drawdown_equal = (cum_returns_equal - cum_max_equal) / cum_max_equal

cum_max_vol = cum_returns_vol.expanding().max()
drawdown_vol = (cum_returns_vol - cum_max_vol) / cum_max_vol

axes[1].fill_between(drawdown_equal.index, drawdown_equal.values, 0,
                     alpha=0.7, color='steelblue', label='Equal-Weight DD')
axes[1].fill_between(drawdown_vol.index, drawdown_vol.values, 0,
                     alpha=0.5, color='green', label='Vol-Weight DD')

axes[1].set_xlabel('Date', fontsize=10)
axes[1].set_ylabel('Drawdown', fontsize=10)
axes[1].set_title('Portfolio Drawdown', fontsize=12, fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].legend(loc='lower left', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 8.2 Comparison vs Single-Stock Strategies

**Key Advantage of Pairs Trading**: Market-neutral (beta ≈ 0)

Let's compare pairs portfolio to:
1. **Long-only S&P stocks** (directional)
2. **Individual pairs** (concentrated risk)
3. **Market index** (beta = 1)

In [ ]:
# Calculate market index (equal-weight all stocks)
returns_wide_ret = prices_wide.pct_change().dropna()
market_returns = returns_wide_ret.mean(axis=1)

# Align dates
market_returns = market_returns.loc[portfolio_returns_equal.index]

# Calculate metrics for market
market_metrics = calculate_performance_metrics(market_returns)

# Calculate beta of pairs portfolio vs market
cov_matrix = np.cov(portfolio_returns_equal, market_returns)
beta = cov_matrix[0, 1] / cov_matrix[1, 1]

print("\n" + "=" * 80)
print("COMPARISON: PAIRS TRADING vs LONG-ONLY MARKET")
print("=" * 80)

print(f"\n{'Metric':<25} {'Pairs Portfolio':<20} {'Market (Long-Only)'}")
print("=" * 80)
print(f"{'Total Return':<25} {equal_weight_metrics['total_return']:<20.2%} {market_metrics['total_return']:.2%}")
print(f"{'Annualized Return':<25} {equal_weight_metrics['annualized_return']:<20.2%} {market_metrics['annualized_return']:.2%}")
print(f"{'Annualized Volatility':<25} {equal_weight_metrics['annualized_vol']:<20.2%} {market_metrics['annualized_vol']:.2%}")
print(f"{'Sharpe Ratio':<25} {equal_weight_metrics['sharpe_ratio']:<20.3f} {market_metrics['sharpe_ratio']:.3f}")
print(f"{'Max Drawdown':<25} {equal_weight_metrics['max_drawdown']:<20.2%} {market_metrics['max_drawdown']:.2%}")
print(f"{'Beta (vs Market)':<25} {beta:<20.3f} {1.000:.3f}")

print("\n💡 Key Insights:")
print(f"  • Pairs portfolio beta = {beta:.3f} → Market-neutral (uncorrelated with market)")
print(f"  • Lower volatility than long-only ({equal_weight_metrics['annualized_vol']:.1%} vs {market_metrics['annualized_vol']:.1%})")
print(f"  • Risk-adjusted returns: Sharpe = {equal_weight_metrics['sharpe_ratio']:.2f} vs {market_metrics['sharpe_ratio']:.2f}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Cumulative returns comparison
cum_returns_pairs = (1 + portfolio_returns_equal).cumprod()
cum_returns_market = (1 + market_returns).cumprod()

axes[0].plot(cum_returns_pairs.index, cum_returns_pairs.values,
            linewidth=2.5, color='green', label='Pairs Portfolio (Market-Neutral)', alpha=0.9)
axes[0].plot(cum_returns_market.index, cum_returns_market.values,
            linewidth=2.5, color='steelblue', label='Market (Long-Only)', alpha=0.7)
axes[0].axhline(1, color='black', linestyle='--', linewidth=1)

axes[0].set_title('Pairs Trading vs Long-Only Market - Cumulative Returns', 
                 fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cumulative Return', fontsize=10)
axes[0].legend(loc='upper left', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Panel 2: Rolling Sharpe ratio (60-day window)
window = 60
rolling_sharpe_pairs = portfolio_returns_equal.rolling(window).mean() / portfolio_returns_equal.rolling(window).std() * np.sqrt(252)
rolling_sharpe_market = market_returns.rolling(window).mean() / market_returns.rolling(window).std() * np.sqrt(252)

axes[1].plot(rolling_sharpe_pairs.index, rolling_sharpe_pairs.values,
            linewidth=2, color='green', label='Pairs Portfolio', alpha=0.9)
axes[1].plot(rolling_sharpe_market.index, rolling_sharpe_market.values,
            linewidth=2, color='steelblue', label='Market', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='-', linewidth=1)
axes[1].axhline(1, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Sharpe = 1.0')

axes[1].set_xlabel('Date', fontsize=10)
axes[1].set_ylabel('Rolling Sharpe Ratio (60-day)', fontsize=10)
axes[1].set_title('Risk-Adjusted Performance Over Time', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Summary: Key Takeaways

### Statistical Arbitrage Pairs Trading

**✓ Methodology**:
1. **Pair Selection**: Correlation screening + cointegration testing (Engle-Granger, Johansen)
2. **Spread Construction**: Hedge ratio from cointegration vector
3. **Signal Generation**: Z-score based entry/exit rules (threshold = 2.0)
4. **Position Sizing**: Dollar-neutral with volatility scaling
5. **Portfolio Diversification**: Multiple pairs reduce idiosyncratic risk
6. **Risk Management**: Half-life monitoring, correlation breakdown detection

**✓ Key Advantages**:
- **Market-Neutral**: Beta ≈ 0 (uncorrelated with market direction)
- **Lower Volatility**: Spread volatility < individual stock volatility
- **Diversification**: Multiple pairs reduce single-pair risk
- **Statistical Edge**: Exploit mean-reversion in cointegrated pairs

**⚠️ Key Risks**:
- **Correlation Breakdown**: Cointegration relationship can change
- **Non-Convergence**: Spreads can diverge beyond stop-loss
- **Model Risk**: Historical cointegration ≠ future cointegration
- **Transaction Costs**: Frequent rebalancing can erode returns

**📊 Typical Performance (Literature)**:
- Sharpe Ratio: 1.0 - 2.0
- Annualized Return: 8% - 15%
- Max Drawdown: 10% - 20%
- Market Correlation: -0.1 to +0.1 (near-zero)

### Production Considerations

1. **Data Quality**: Use high-quality price data, handle splits/dividends
2. **Transaction Costs**: Include commissions, bid-ask spreads, slippage
3. **Execution**: Use limit orders to reduce slippage
4. **Monitoring**: Real-time correlation tracking, automated stop-losses
5. **Rebalancing**: Test cointegration monthly, update hedge ratios

### Next Steps

- **Backtest on real data**: Test on S&P 500, sector ETFs, or currency pairs
- **Add transaction costs**: Realistic cost model (2-5 bps per trade)
- **Optimize parameters**: Grid search for z-score thresholds, lookback periods
- **Advanced techniques**: Kalman filters for dynamic hedge ratios, regime switching
- **Integration**: Combine with other strategies (momentum, value) for alpha diversification

---

**Paper References**:
- Engle & Granger (1987): "Co-integration and Error Correction: Representation, Estimation, and Testing"
- Johansen (1988): "Statistical Analysis of Cointegration Vectors"
- Gatev, Goetzmann & Rouwenhorst (2006): "Pairs Trading: Performance of a Relative-Value Arbitrage Rule"
- Do & Faff (2010): "Does Simple Pairs Trading Still Work?"
- Avellaneda & Lee (2010): "Statistical Arbitrage in the U.S. Equities Market"

**2025 Research**:
- Pre-selection in cointegration-based pairs trading (Springer 2023)
- Copula-based trading of cointegrated cryptocurrency pairs (2024)
- 7 Innovative Pairs Trading Strategies for 2025 (ChartsWatcher)